In [ ]:
import cv2
import numpy as np
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
import threading
import time

# Playback control flags
video_running = False
video_paused = False
video_thread = None
current_cap = None
video_canvas = None


classmates = {
    "abdullah": {"matno": "21120613126"},
    "ada": {"matno": "21120612451"},
    "chima": {"matno": "21120612567"},
    "clement": {"matno": "21120612479"},
    "daniel": {"matno": "21120612478"},
    "emeka": {"matno": "21120612449"},
    "ilyas": {"matno": "21120612448"},
    "miracle": {"matno": "21120612475"},
    "tabitha": {"matno": "21120612496"},
    "tireni": {"matno": "21120613127"},
}

# LOAD YOLO MODEL
net = cv2.dnn.readNet('C:/Users/abdul/Desktop/School/Year 4/Year 4 Semester 2/CSC 418/Abdullah_CSC418/Week_5/cfg/yolov3.weights',
                      'C:/Users/abdul/Desktop/School/Year 4/Year 4 Semester 2/CSC 418/Abdullah_CSC418/Week_5/cfg/yolov3.cfg')

# Load class labels
with open('C:/Users/abdul/Desktop/School/Year 4/Year 4 Semester 2/CSC 418/Abdullah_CSC418/Week_5/cfg/coco.names', 'r') as f:
    classes = f.read().splitlines()

# Generate random colors for each class
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# GUI SETUP
window = tk.Tk()
window.title("YOLO Object Detection")
window.geometry('500x500')
window.configure(bg='#333333')

# Store current user
current_user = None

# LOGIN LOGIC
def login():
    username = username_entry.get().strip().lower()
    password = password_entry.get()

    if username in classmates and password == classmates[username]["matno"]:
        global current_user
        current_user = username
        messagebox.showinfo("Login Success", f"Welcome, {username.title()}!")
        show_video_frame()
    else:
        messagebox.showerror("Login Failed", "Invalid username or matric number.")

# LOGIN SCREEN
def show_login_frame():
    for widget in window.winfo_children():
        widget.destroy()

    frame = tk.Frame(window, bg='#333333')

    # Title
    tk.Label(frame, text="Login", bg='#333333', fg="#4338CA", font=("Arial", 24)).grid(row=0, column=0, columnspan=2, pady=20)

    # Username
    tk.Label(frame, text="Username:", bg='#333333', fg="#FFFFFF", font=("Arial", 14)).grid(row=1, column=0, padx=10, pady=10, sticky='e')
    global username_entry
    username_entry = tk.Entry(frame, font=("Arial", 14))
    username_entry.grid(row=1, column=1, padx=10, pady=10)

    # Matric Number (password field)
    tk.Label(frame, text="Matric Number:", bg='#333333', fg="#FFFFFF", font=("Arial", 14)).grid(row=2, column=0, padx=10, pady=10, sticky='e')
    global password_entry
    password_entry = tk.Entry(frame, font=("Arial", 14), show="*")
    password_entry.grid(row=2, column=1, padx=10, pady=10)

    # Login Button
    tk.Button(frame, text="Login", bg="#4338CA", fg="#FFFFFF", font=("Arial", 16), command=login).grid(row=3, column=0, columnspan=2, pady=30)

    frame.pack()

# VIDEO SELECTION SCREEN
def show_video_frame():
    for widget in window.winfo_children():
        widget.destroy()

    frame = tk.Frame(window, bg='#333333')
    frame.pack()

    tk.Label(frame, text="YOLO Object Detection", bg='#333333', fg="#4338CA", font=("Arial", 20)).pack(pady=10)

    # Canvas for displaying video
    global video_canvas
    video_canvas = tk.Label(frame)
    video_canvas.pack()

    # Video selection buttons
    video_btns = tk.Frame(frame, bg='#333333')
    video_btns.pack(pady=10)
    tk.Button(video_btns, text="Mosque", bg="#4338CA", fg="white", font=("Arial", 14),
              command=lambda: start_video_thread("videos/Mosque.mp4")).grid(row=0, column=0, padx=5)
    tk.Button(video_btns, text="Cafeteria", bg="#4338CA", fg="white", font=("Arial", 14),
              command=lambda: start_video_thread("videos/Cafeteria.mp4")).grid(row=0, column=1, padx=5)
    tk.Button(video_btns, text="TYD Lab 2", bg="#4338CA", fg="white", font=("Arial", 14),
              command=lambda: start_video_thread("videos/TYD Lab 2.mp4")).grid(row=0, column=2, padx=5)

    # Playback control buttons
    control_btns = tk.Frame(frame, bg='#333333')
    control_btns.pack(pady=10)
    tk.Button(control_btns, text="Play", bg="green", fg="white", font=("Arial", 14), command=resume_video).grid(row=0, column=0, padx=5)
    tk.Button(control_btns, text="Pause", bg="orange", fg="white", font=("Arial", 14), command=pause_video).grid(row=0, column=1, padx=5)
    #tk.Button(control_btns, text="Quit", bg="red", fg="white", font=("Arial", 14), command=stop_video).grid(row=0, column=2, padx=5)

    # Back Button
    tk.Button(frame, text="Back to Login", bg="gray", fg="white", font=("Arial", 12), command=show_login_frame).pack(pady=10)



# YOLO VIDEO PROCESSING FUNCTION
def run_yolo(video_path):
    global video_running, video_paused

    cap = cv2.VideoCapture(video_path)

    while video_running:
        if video_paused:
            time.sleep(0.1)
            continue

        ret, frame = cap.read()
        if not ret:
            break

        height, width, _ = frame.shape

        # YOLO preprocessing
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
        net.setInput(blob)
        output_layers_names = net.getUnconnectedOutLayersNames()
        layer_outputs = net.forward(output_layers_names)

        boxes = []
        confidences = []
        class_ids = []

        for output in layer_outputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

        if len(indexes) > 0:
            for i in indexes.flatten():
                x, y, w, h = boxes[i]
                label = str(classes[class_ids[i]])
                confidence = str(round(confidences[i], 2))
                color = colors[class_ids[i]]
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
                cv2.putText(frame, f"{label} {confidence}", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        # Convert frame for Tkinter
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb_frame)
        imgtk = ImageTk.PhotoImage(image=img)

        # Update Tkinter label
        video_canvas.imgtk = imgtk
        video_canvas.configure(image=imgtk)

        time.sleep(0.03)

    cap.release()



def start_video_thread(path):
    global video_running, video_paused, video_thread

    # If a video is already running, stop it first
    if video_running:
        stop_video()
        if video_thread is not None:
            video_thread.join()  # wait for previous thread to fully stop

    video_running = True
    video_paused = False
    video_thread = threading.Thread(target=run_yolo, args=(path,))
    video_thread.start()


def pause_video():
    global video_paused
    video_paused = True

def resume_video():
    global video_paused
    video_paused = False

def stop_video():
    global video_running, video_paused
    video_running = False
    video_paused = False


show_login_frame()
window.mainloop()